# Interactive visualisation with plotly

This notebook shows the drawing functions in `calfem.vis_plotly`. The plots are interactive:

* **hover** an element to see its number and value,
* **drag** to zoom, **double click** to reset the view,
* use the toolbar in the upper right corner to pan or save a PNG.

`vis_plotly` has the same functions as `vis_mpl`, so existing CALFEM scripts can switch to interactive plots by changing one import.

Two problems are solved on the same geometry, a plate with a hole:

1. **Plane stress** – the plate in tension, using 4-node elements (`plani4e`/`plani4s`).
2. **Heat flow** – a temperature difference between the short sides, using 3-node elements (`flw2te`/`flw2ts`).

## Installation

In Google Colab or a new environment, install CALFEM with the plotly extra by uncommenting the line below.

In [ ]:
# !pip install calfem-python[plotly]

In [ ]:
import numpy as np

import calfem.core as cfc
import calfem.geometry as cfg
import calfem.mesh as cfm
import calfem.utils as cfu
import calfem.vis_plotly as cfv

## Geometry

A 0.4 × 0.2 m plate with a hole of radius 0.04 m. The left side gets marker 10 and the right side marker 20, used for boundary conditions and loads.

In [ ]:
L = 0.4             # Plate length [m]
H = 0.2             # Plate height [m]
r = 0.04            # Hole radius [m]

left_marker = 10
right_marker = 20

g = cfg.Geometry()

g.point([0.0, 0.0])         # 0
g.point([L, 0.0])           # 1
g.point([L, H])             # 2
g.point([0.0, H])           # 3

g.point([L/2, H/2])         # 4 - hole center
g.point([L/2 + r, H/2])     # 5
g.point([L/2, H/2 + r])     # 6
g.point([L/2 - r, H/2])     # 7
g.point([L/2, H/2 - r])     # 8

g.spline([0, 1])                        # 0
g.spline([1, 2], marker=right_marker)   # 1
g.spline([2, 3])                        # 2
g.spline([3, 0], marker=left_marker)    # 3

g.circle([5, 4, 6])                     # 4
g.circle([6, 4, 7])                     # 5
g.circle([7, 4, 8])                     # 6
g.circle([8, 4, 5])                     # 7

g.surface([0, 1, 2, 3], holes=[[4, 5, 6, 7]])

`draw_geometry` shows point and curve numbers, with markers in brackets. Hover a point to see its coordinates.

In [ ]:
cfv.figure()
cfv.draw_geometry(g, title="Geometry")
cfv.show()

## 1. Plane stress

### Mesh

In [ ]:
mesh = cfm.GmshMesh(g)
mesh.el_type = 3            # 4-node quadrilaterals
mesh.dofs_per_node = 2
mesh.el_size_factor = 0.015

coords, edof, dofs, bdofs, elementmarkers = mesh.create()

print(f"{edof.shape[0]} elements, {coords.shape[0]} nodes")

cfv.figure()
cfv.draw_mesh(coords, edof, mesh.dofs_per_node, mesh.el_type, filled=True, title="Mesh")
cfv.show()

### Solve

The left side is fixed and a total force of 400 kN is distributed over the nodes on the right side.

In [ ]:
t = 0.01            # Thickness [m]
E = 210e9           # Young's modulus [Pa]
v = 0.3             # Poisson's ratio
F = 4e5             # Total tensile force [N]

ep = [1, t, 2]      # Plane stress, thickness, 2x2 Gauss points
D = cfc.hooke(1, E, v)

n_dofs = np.size(dofs)
ex, ey = cfc.coordxtr(edof, coords, dofs)

K = np.zeros([n_dofs, n_dofs])
for eltopo, elx, ely in zip(edof, ex, ey):
    Ke, _ = cfc.plani4e(elx, ely, ep, D)
    cfc.assem(eltopo, K, Ke)

f = np.zeros([n_dofs, 1])
bc = np.array([], "i")
bc_val = np.array([], "f")
bc, bc_val = cfu.apply_bc(bdofs, bc, bc_val, left_marker, 0.0, 0)

n_right_nodes = len(bdofs[right_marker]) // mesh.dofs_per_node
cfu.apply_force(bdofs, f, right_marker, F/n_right_nodes, 1)

a, r = cfc.solveq(K, f, bc, bc_val)

### Effective stress

`plani4s` gives the stresses in the four Gauss points of each element. They are averaged to one von Mises stress per element.

In [ ]:
ed = cfc.extract_eldisp(edof, a)
von_mises = np.zeros(edof.shape[0])

for i, (elx, ely, eld) in enumerate(zip(ex, ey, ed)):
    es, et = cfc.plani4s(elx, ely, ep, D, eld)
    sigx, sigy, tauxy = np.mean(es, axis=0)
    von_mises[i] = np.sqrt(sigx**2 - sigx*sigy + sigy**2 + 3*tauxy**2)

print(f"Max effective stress: {von_mises.max()/1e6:.0f} MPa")

`draw_element_values` colors each element by its value. Here the elements are drawn in the displaced configuration, magnified 50 times. Hover the elements next to the hole to find the highest stress.

In [ ]:
cfv.figure()
cfv.draw_element_values(
    von_mises/1e6, coords, edof, mesh.dofs_per_node, mesh.el_type,
    displacements=a, magnfac=50.0, draw_elements=False,
    title="Effective stress (displacements x 50)",
)
cfv.colorbar("MPa")
cfv.show()

`draw_displacements` draws the displaced mesh. Without `magnfac` the displacements are scaled so that the largest one is 10 % of the model size.

In [ ]:
cfv.figure()
cfv.draw_displacements(a, coords, edof, mesh.dofs_per_node, mesh.el_type,
                       draw_undisplaced_mesh=True, title="Displacements")
cfv.show()

## 2. Heat flow

The same geometry meshed with triangles, 100 °C on the left side and 20 °C on the right side.

In [ ]:
k = 50.0            # Thermal conductivity [W/(m K)]
D_heat = k*np.eye(2)
ep_heat = [t]

mesh = cfm.GmshMesh(g)
mesh.el_type = 2            # 3-node triangles
mesh.dofs_per_node = 1
mesh.el_size_factor = 0.02

coords, edof, dofs, bdofs, elementmarkers = mesh.create()

n_dofs = np.size(dofs)
ex, ey = cfc.coordxtr(edof, coords, dofs)

K = np.zeros([n_dofs, n_dofs])
for eltopo, elx, ely in zip(edof, ex, ey):
    Ke = cfc.flw2te(elx, ely, ep_heat, D_heat)
    cfc.assem(eltopo, K, Ke)

f = np.zeros([n_dofs, 1])
bc = np.array([], "i")
bc_val = np.array([], "f")
bc, bc_val = cfu.apply_bc(bdofs, bc, bc_val, left_marker, 100.0, 0)
bc, bc_val = cfu.apply_bc(bdofs, bc, bc_val, right_marker, 20.0, 0)

T, Q = cfc.solveq(K, f, bc, bc_val)

ed = cfc.extract_eldisp(edof, T)
flux = np.zeros((edof.shape[0], 2))
for i, (elx, ely, eld) in enumerate(zip(ex, ey, ed)):
    es, et = cfc.flw2ts(elx, ely, D_heat, eld)
    flux[i, :] = np.ravel(es)

### Temperature

The nodal temperatures can be drawn as filled contours, contour lines or smoothly shaded. Hover anywhere in the plate to read the temperature.

In [ ]:
cfv.figure()
cfv.draw_nodal_values_contourf(
    T, coords, edof, levels=16,
    dofs_per_node=mesh.dofs_per_node, el_type=mesh.el_type, draw_elements=True,
    title="Temperature", colorscale="RdBu_r",
)
cfv.colorbar("°C")
cfv.show()

In [ ]:
cfv.figure()
cfv.draw_nodal_values_contour(T, coords, edof, levels=16, title="Isotherms", colorscale="RdBu_r")
cfv.show()

cfv.figure()
cfv.draw_nodal_values_shaded(T, coords, edof, title="Temperature, shaded", colorscale="RdBu_r")
cfv.show()

### Heat flux

`draw_element_flux` draws one arrow per element. Hover an arrow to see the flux components.

In [ ]:
cfv.figure()
cfv.draw_element_flux(flux, coords, edof, mesh.dofs_per_node, mesh.el_type,
                      color_by_magnitude=True, draw_elements=True, title="Heat flux")
cfv.colorbar("W/m²")
cfv.show()

## 3D meshes

`draw_mesh` and `draw_element_values` also handle 3D meshes of tetrahedra and hexahedra. Drag to rotate the model. Here a small block of hexahedra is created directly and colored by the x-coordinate of the element centers.

In [ ]:
nx, ny, nz = 6, 3, 2
X, Y, Z = np.meshgrid(np.arange(nx + 1), np.arange(ny + 1), np.arange(nz + 1), indexing="ij")
coords3 = np.column_stack((X.ravel(), Y.ravel(), Z.ravel())).astype(float)

def node(i, j, k):
    return (i*(ny + 1) + j)*(nz + 1) + k

nodes = np.array([
    [node(i, j, k), node(i+1, j, k), node(i+1, j+1, k), node(i, j+1, k),
     node(i, j, k+1), node(i+1, j, k+1), node(i+1, j+1, k+1), node(i, j+1, k+1)]
    for i in range(nx) for j in range(ny) for k in range(nz)
])
dofs3 = np.arange(1, 3*coords3.shape[0] + 1).reshape(-1, 3)
edof3 = dofs3[nodes].reshape(nodes.shape[0], -1)

x_center = coords3[nodes, 0].mean(axis=1)

cfv.figure()
cfv.draw_element_values(x_center, coords3, edof3, 3, 5, title="Hexahedra")
cfv.show()

## Saving and customising figures

`save_figure` writes the current figure as an interactive web page (`.html`) or as an image (`.png`, `.svg`, `.pdf`, which requires the `kaleido` package). `gcf()` returns the plotly figure for further customisation with the plotly API.

In [ ]:
cfv.figure()
cfv.draw_element_flux(flux, coords, edof, mesh.dofs_per_node, mesh.el_type, title="Heat flux")

fig = cfv.gcf()
fig.update_layout(width=700, height=400)

cfv.save_figure("exm_plotly_flux.html", include_plotlyjs="cdn")
cfv.show()

## Switching back to matplotlib

The same calls work with `calfem.vis_mpl` for static figures, e.g. for reports:

```python
import calfem.vis_mpl as cfv

cfv.figure()
cfv.draw_nodal_values_contourf(T, coords, edof, levels=16, title="Temperature")
cfv.colorbar()
cfv.show_and_wait()
```